# Lasso Logistic Regression xG Model

Trains and evaluates an L1-penalised (Lasso) logistic regression expected-goals model on StatsBomb open data.

The key property of Lasso is **sparsity**: as regularisation increases, coefficients are driven to exactly zero, performing automatic feature selection. This makes Lasso useful for understanding which features are truly necessary for xG prediction versus which are redundant.

**Pipeline summary:**
1. Load data and engineer features (reuses `src.features`)
2. Preprocess — standardise numerics, encode categoricals, impute
3. Tune regularisation strength `C` with Optuna (5-fold CV, log-loss)
4. Final cross-validated evaluation — AUC, log-loss, Brier score
5. Calibration curve
6. Regularisation path — how sparsity evolves across `C` values
7. Surviving feature coefficients at the best `C`
8. Baseline comparison (Ridge vs Lasso vs distance-only)
9. Save model to `outputs/`

In [ ]:
import sys, warnings, os
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from statsbombpy import sb

from src.features import create_xg_features

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
os.makedirs('../outputs', exist_ok=True)

## 1. Data Loading and Feature Engineering

In [ ]:
COMPETITIONS = [
    {'competition_id': 11, 'season_id': 27,  'label': 'La Liga 2015/16'},
    {'competition_id': 16, 'season_id': 4,   'label': 'Champions League 2018/19'},
    {'competition_id': 2,  'season_id': 27,  'label': 'Premier League 2015/16'},
    {'competition_id': 43, 'season_id': 106, 'label': 'World Cup 2022'},
]

def load_competition(competition_id, season_id):
    matches = sb.matches(competition_id=competition_id, season_id=season_id)
    return pd.concat(
        [sb.events(match_id=mid).assign(match_id=mid) for mid in matches['match_id']],
        ignore_index=True,
    )

all_events = []
for comp in COMPETITIONS:
    print(f"Loading {comp['label']}…")
    all_events.append(load_competition(comp['competition_id'], comp['season_id']))

events_df = pd.concat(all_events, ignore_index=True)
xg_df = create_xg_features(events_df)

shots = xg_df[xg_df['is_penalty'] == 0].copy().reset_index(drop=True)

print(f"\nShots (excl. penalties): {len(shots):,}")
print(f"Goals: {shots['is_goal'].sum():,} ({shots['is_goal'].mean():.1%})")

## 2. Preprocessing

Identical to the Ridge notebook — Lasso is also scale-sensitive, so `StandardScaler` is required. The `saga` solver is used instead of `lbfgs` because `lbfgs` does not support L1 regularisation.

In [ ]:
CATEGORICAL_FEATURES = ['shot_body_part', 'shot_technique', 'previous_event_type']

NUMERIC_FEATURES = [
    'distance_to_goal', 'shot_angle', 'centrality',
    'in_penalty_area', 'in_six_yard_box',
    'shot_first_time', 'shot_under_pressure',
    'num_defenders_in_frame', 'num_teammates_in_frame',
    'distance_to_nearest_defender', 'num_defenders_between_shot_and_goal',
    'goalkeeper_distance_to_goal', 'goalkeeper_distance_to_shooter',
    'previous_event_was_pass', 'previous_event_was_carry',
    'previous_event_same_team', 'previous_event_distance',
    'is_late_game', 'is_extra_time',
]

TARGET = 'is_goal'

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ]), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), CATEGORICAL_FEATURES),
], remainder='drop')

X = shots[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = shots[TARGET].values

X_transformed = preprocessor.fit_transform(X)
ohe_feature_names = (
    preprocessor.named_transformers_['cat']['ohe']
    .get_feature_names_out(CATEGORICAL_FEATURES)
    .tolist()
)
ALL_FEATURE_NAMES = NUMERIC_FEATURES + ohe_feature_names
print(f"Feature matrix: {X_transformed.shape[0]:,} shots × {X_transformed.shape[1]} features")

## 3. Hyperparameter Tuning (Optuna)

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    C = trial.suggest_float('C', 1e-4, 1e3, log=True)
    model = Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(
            penalty='l1', C=C, solver='saga',
            max_iter=2000, random_state=42,
        )),
    ])
    fold_losses = []
    for train_idx, val_idx in CV.split(X, y):
        model.fit(X.iloc[train_idx], y[train_idx])
        proba = model.predict_proba(X.iloc[val_idx])[:, 1]
        fold_losses.append(log_loss(y[val_idx], proba))
    return np.mean(fold_losses)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, show_progress_bar=True)

best_C = study.best_params['C']
print(f"\nBest log-loss: {study.best_value:.4f}")
print(f"Best C:        {best_C:.5f}")

In [ ]:
trials_df = study.trials_dataframe().sort_values('params_C')

fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogx(trials_df['params_C'], trials_df['value'], 'o', alpha=0.6, ms=5, color='#4C72B0')
ax.axvline(best_C, color='#DD8452', linewidth=1.5, linestyle='--', label=f'Best C={best_C:.4f}')
ax.set_xlabel('C (log scale)')
ax.set_ylabel('CV log-loss')
ax.set_title('Regularisation strength search')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Final Cross-Validated Evaluation

In [ ]:
final_model = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(
        penalty='l1', C=best_C, solver='saga',
        max_iter=2000, random_state=42,
    )),
])

oof_proba = cross_val_predict(final_model, X, y, cv=CV, method='predict_proba')[:, 1]

metrics = {
    'ROC-AUC':     roc_auc_score(y, oof_proba),
    'Log-loss':    log_loss(y, oof_proba),
    'Brier score': brier_score_loss(y, oof_proba),
}

print("Out-of-fold evaluation metrics:")
for name, val in metrics.items():
    print(f"  {name:<15} {val:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

RocCurveDisplay.from_predictions(y, oof_proba, ax=axes[0], color='#4C72B0')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_title('ROC curve (out-of-fold)')

PrecisionRecallDisplay.from_predictions(y, oof_proba, ax=axes[1], color='#DD8452')
axes[1].axhline(y.mean(), color='k', linestyle='--', linewidth=0.8, label=f'Baseline ({y.mean():.3f})')
axes[1].legend()
axes[1].set_title('Precision-Recall curve (out-of-fold)')

plt.tight_layout()
plt.show()

## 5. Calibration

In [ ]:
fraction_pos, mean_pred = calibration_curve(y, oof_proba, n_bins=15, strategy='quantile')

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Perfect calibration')
ax.plot(mean_pred, fraction_pos, 'o-', color='#4C72B0', linewidth=2, markersize=6, label='Lasso xG')
ax.set_xlabel('Mean predicted xG')
ax.set_ylabel('Fraction of goals')
ax.set_title('Calibration curve')
ax.legend()
ax.set_xlim(0, 0.7)
ax.set_ylim(0, 0.7)
plt.tight_layout()
plt.show()

bins = np.linspace(0, 1, 11)
bin_ids = np.digitize(oof_proba, bins) - 1
ece = sum(
    (np.sum(bin_ids == b) / len(y)) * abs(y[bin_ids == b].mean() - oof_proba[bin_ids == b].mean())
    for b in range(len(bins) - 1) if np.sum(bin_ids == b) > 0
)
print(f"Expected Calibration Error (ECE): {ece:.4f}")

## 6. Regularisation Path

As `C` decreases (stronger L1 penalty), features are eliminated one by one until only the most predictive remain. This path shows which features the model considers essential and which are redundant.

Each line is one feature; when it hits zero it has been excluded by Lasso.

In [ ]:
C_values = np.logspace(-3, 2, 50)
coef_path = []
n_active_path = []

X_proc = preprocessor.transform(X)

for C in C_values:
    clf = LogisticRegression(
        penalty='l1', C=C, solver='saga',
        max_iter=2000, random_state=42,
    )
    clf.fit(X_proc, y)
    coef_path.append(clf.coef_[0].copy())
    n_active_path.append(np.sum(clf.coef_[0] != 0))

coef_path = np.array(coef_path)   # shape (n_C, n_features)
n_active_path = np.array(n_active_path)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

# Top: coefficient paths for every feature
for i in range(coef_path.shape[1]):
    axes[0].semilogx(C_values, coef_path[:, i], linewidth=0.8, alpha=0.6)
axes[0].axvline(best_C, color='black', linewidth=1.2, linestyle='--', label=f'Best C={best_C:.4f}')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_ylabel('Coefficient')
axes[0].set_title('Lasso regularisation path — all features')
axes[0].legend(fontsize=9)

# Bottom: number of active (non-zero) features
axes[1].semilogx(C_values, n_active_path, color='#4C72B0', linewidth=2)
axes[1].axvline(best_C, color='black', linewidth=1.2, linestyle='--')
axes[1].set_xlabel('C (log scale) — left = more regularisation')
axes[1].set_ylabel('Active features')
axes[1].set_title('Number of non-zero coefficients vs regularisation')
axes[1].set_ylim(0, coef_path.shape[1] + 1)

# Annotate active count at best_C
best_idx = np.argmin(np.abs(C_values - best_C))
n_best = n_active_path[best_idx]
axes[1].annotate(
    f'{n_best} features\nat best C',
    xy=(best_C, n_best), xytext=(best_C * 3, n_best - 3),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=9,
)

plt.tight_layout()
plt.show()
print(f"Active features at best C ({best_C:.4f}): {n_best} / {coef_path.shape[1]}")

## 7. Surviving Feature Coefficients

In [ ]:
final_model.fit(X, y)
coefs = pd.Series(
    final_model.named_steps['clf'].coef_[0],
    index=ALL_FEATURE_NAMES,
)

zeroed   = coefs[coefs == 0].index.tolist()
survived = coefs[coefs != 0].sort_values()

print(f"Total features:   {len(coefs)}")
print(f"Non-zero (kept):  {len(survived)}")
print(f"Zeroed out:       {len(zeroed)}")
print(f"\nEliminated features:")
for f in zeroed:
    print(f"  {f}")

In [ ]:
colors = ['#DD8452' if v > 0 else '#4C72B0' for v in survived]

fig, ax = plt.subplots(figsize=(8, max(4, len(survived) * 0.28)))
survived.plot.barh(ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (log-odds per SD)')
ax.set_title(f'Lasso — {len(survived)} surviving feature coefficients at best C')
plt.tight_layout()
plt.show()

In [ ]:
# Odds ratios for surviving features
odds_ratios = np.exp(survived).sort_values()
colors_or = ['#DD8452' if v > 1 else '#4C72B0' for v in odds_ratios]

fig, ax = plt.subplots(figsize=(8, max(4, len(survived) * 0.28)))
odds_ratios.plot.barh(ax=ax, color=colors_or, edgecolor='white')
ax.axvline(1, color='black', linewidth=0.8)
ax.set_xlabel('Odds ratio')
ax.set_title('Lasso — odds ratios for surviving features')
plt.tight_layout()
plt.show()

## 8. Model Comparison

Comparing Lasso against Ridge and a distance-only baseline highlights the trade-off between sparsity and predictive performance.

In [ ]:
# Distance-only baseline
baseline_model = Pipeline([
    ('pre', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ])),
    ('clf', LogisticRegression(penalty='l2', C=1.0, solver='lbfgs',
                               max_iter=1000, random_state=42)),
])
baseline_proba = cross_val_predict(
    baseline_model, shots[['distance_to_goal', 'shot_angle']], y,
    cv=CV, method='predict_proba',
)[:, 1]

# Ridge (reload saved model OOF probabilities would require re-running;
# re-compute here so this notebook is self-contained)
ridge_model = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(penalty='l2', C=best_C, solver='lbfgs',
                               max_iter=1000, random_state=42)),
])
ridge_proba = cross_val_predict(ridge_model, X, y, cv=CV, method='predict_proba')[:, 1]

def eval_metrics(y_true, proba):
    return {
        'ROC-AUC':     roc_auc_score(y_true, proba),
        'Log-loss':    log_loss(y_true, proba),
        'Brier score': brier_score_loss(y_true, proba),
    }

comparison = pd.DataFrame({
    'Distance + angle': eval_metrics(y, baseline_proba),
    'Ridge':            eval_metrics(y, ridge_proba),
    'Lasso':            metrics,
}).T.round(4)

print("Model comparison (out-of-fold, same CV splits):")
print(comparison.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for proba, name, color in [
    (baseline_proba, f"Distance+angle  (AUC={eval_metrics(y, baseline_proba)['ROC-AUC']:.3f})", '#55A868'),
    (ridge_proba,    f"Ridge           (AUC={eval_metrics(y, ridge_proba)['ROC-AUC']:.3f})",    '#4C72B0'),
    (oof_proba,      f"Lasso           (AUC={metrics['ROC-AUC']:.3f})",                          '#DD8452'),
]:
    RocCurveDisplay.from_predictions(y, proba, ax=ax, color=color, name=name)

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_title('ROC comparison — baseline vs Ridge vs Lasso')
plt.tight_layout()
plt.show()

## 9. Save Model

In [ ]:
model_path = '../outputs/lasso_xg_model.pkl'
joblib.dump(final_model, model_path)
print(f"Model saved to {model_path}")

reloaded = joblib.load(model_path)
check = reloaded.predict_proba(X.iloc[:5])[:, 1]
ref   = final_model.predict_proba(X.iloc[:5])[:, 1]
assert np.allclose(check, ref), "Reloaded model predictions differ!"
print("Reload check passed.")

## 10. Results Summary

In [ ]:
print("="*45)
print(" Lasso Logistic Regression xG Model")
print("="*45)
print(f" Dataset:          {len(shots):,} open-play shots")
print(f" Goal rate:        {y.mean():.1%}")
print(f" Total features:   {len(ALL_FEATURE_NAMES)}")
print(f" Best C:           {best_C:.5f}")
print(f" Surviving feats:  {len(survived)} / {len(ALL_FEATURE_NAMES)}")
print(f" CV folds:         5-fold stratified")
print("-"*45)
for name, val in metrics.items():
    print(f" {name:<15} {val:.4f}")
print(f" {'ECE':<15} {ece:.4f}")
print("-"*45)
print(f" Model saved:      {model_path}")
print("="*45)